# 02. Geração de Pseudo-Labels com DeepForest

Este notebook executa o modelo pré-treinado DeepForest (RetinaNet) sobre as imagens compactadas no arquivo HDF5, converte as caixas para o formato YOLO [class_id, x_center, y_center, w, h] normalizado, e salva as anotações geradas no subgrupo `labels` correspondente.

**Responsável:** Pessoa 3

In [2]:
import sys
sys.path.append('..')

import utils
import h5py
import numpy as np
import cv2

In [5]:
import h5py

HDF5_PATH = "dataset_v1_raw.h5"

hdf5 = h5py.File(HDF5_PATH, "r")
images = hdf5["images"]

print("OK carregado:", len(images))

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'dataset_v1_raw.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

## 1. Carregamento do HDF5 e do Modelo DeepForest

In [12]:
class FakeModel:
    def predict_image(self, image, return_plot=False):
        import pandas as pd

        return pd.DataFrame([
            {
                "xmin": 100,
                "ymin": 100,
                "xmax": 300,
                "ymax": 300,
                "score": 0.95
            }
        ])

model = FakeModel()

print("DeepForest MOCK ativo (ambiente liberado)")

DeepForest MOCK ativo (ambiente liberado)


## 2. Predição de Caixas e Normalização YOLO (Pessoa 3)

Conversão de [xmin, ymin, xmax, ymax] em pixels para as coordenadas normalizadas [0.0, 1.0].

In [13]:
def convert_to_yolo(box, img_w, img_h):
    """
    box: [xmin, ymin, xmax, ymax]
    retorna: [x_center, y_center, width, height] normalizado
    """
    xmin, ymin, xmax, ymax = box

    x_center = (xmin + xmax) / 2.0 / img_w
    y_center = (ymin + ymax) / 2.0 / img_h
    width = (xmax - xmin) / img_w
    height = (ymax - ymin) / img_h

    return [x_center, y_center, width, height]


pseudo_labels = []

for i in range(len(images)):
    img = images[i]

    if img.dtype != np.uint8:
        img = img.astype(np.uint8)

    preds = model.predict_image(image=img, return_plot=False)

    img_h, img_w = img.shape[:2]

    if preds is not None and len(preds) > 0:
        for _, row in preds.iterrows():
            box = [
                row["xmin"],
                row["ymin"],
                row["xmax"],
                row["ymax"]
            ]

            yolo_box = convert_to_yolo(box, img_w, img_h)

            pseudo_labels.append({
                "image_id": i,
                "class": 0, 
                "bbox": yolo_box,
                "score": row["score"]
            })

print("Pseudo-labels geradas:", len(pseudo_labels))

Pseudo-labels geradas: 0


## 3. Gravação das Pseudo-Labels no HDF5

In [14]:
if "labels" in hdf5:
    del hdf5["labels"]

label_group = hdf5.create_group("labels")

image_ids = []
classes = []
bboxes = []
scores = []

for p in pseudo_labels:
    image_ids.append(p["image_id"])
    classes.append(p["class"])
    bboxes.append(p["bbox"])
    scores.append(p["score"])

label_group.create_dataset("image_id", data=np.array(image_ids))
label_group.create_dataset("class", data=np.array(classes))
label_group.create_dataset("bbox", data=np.array(bboxes))
label_group.create_dataset("score", data=np.array(scores))

hdf5.flush()
hdf5.close()

print("Pseudo-labels salvas no HDF5 com sucesso")

Pseudo-labels salvas no HDF5 com sucesso


## 4. Criando os arquivos HDF5 de treino e validação

In [3]:
from utils import load_tile_mapping
from utils import create_geographical_split
from utils import create_hdf5_subset

mapping = load_tile_mapping("../data/tile_mapping.csv")
print(mapping.head())

  tile_hdf5              sector                       tile_filename
0    tile_0  asa_norte_setor_01     asa_norte_setor_01_tile_0_0.tif
1    tile_1  asa_norte_setor_01  asa_norte_setor_01_tile_0_1280.tif
2    tile_2  asa_norte_setor_01  asa_norte_setor_01_tile_0_1920.tif
3    tile_3  asa_norte_setor_01   asa_norte_setor_01_tile_0_640.tif
4    tile_4  asa_norte_setor_01  asa_norte_setor_01_tile_1280_0.tif


In [4]:
train_tiles, val_tiles = create_geographical_split(
    "../data/tile_mapping.csv"
)

print(len(train_tiles))
print(len(val_tiles))

64
32


In [16]:
HDF5_PATH = "../data/dataset_v1_raw.h5"

TRAIN_OUTPUT = "../data/dataset_treino.h5"

VAL_OUTPUT = "../data/dataset_val.h5"

## 4.1 Dataset de treino

In [17]:
create_hdf5_subset(

    input_hdf5=HDF5_PATH,

    output_hdf5=TRAIN_OUTPUT,

    tile_names=train_tiles,

    mapping_df=mapping,

    subset_name="train",

)

PosixPath('../data/dataset_treino.h5')

## 4.2 Dataset de validação

In [18]:
create_hdf5_subset(

    input_hdf5=HDF5_PATH,

    output_hdf5=VAL_OUTPUT,

    tile_names=val_tiles,

    mapping_df=mapping,

    subset_name="validation",

)

PosixPath('../data/dataset_val.h5')

## 4.3 Verificando arquivos HDF5

In [22]:
for dataset in [TRAIN_OUTPUT, VAL_OUTPUT]:

    with h5py.File(dataset, "r") as f:

        print("=" * 60)

        print(dataset)

        print()

        print("Atributos:")

        print(dict(f.attrs))

        print()

        print("Número de imagens:")

        print(len(f["images"]))

        print()

        print("Número de labels:")

        print(len(f["labels"]))

../data/dataset_treino.h5

Atributos:
{'sectors': 'asa_norte_setor_01,asa_norte_setor_02,asa_norte_setor_03,asa_norte_setor_04', 'source_dataset': 'dataset_v1_raw.h5', 'subset': 'train', 'total_tiles': np.int64(64)}

Número de imagens:
64

Número de labels:
64
../data/dataset_val.h5

Atributos:
{'sectors': 'asa_norte_setor_05,asa_norte_setor_06', 'source_dataset': 'dataset_v1_raw.h5', 'subset': 'validation', 'total_tiles': np.int64(32)}

Número de imagens:
32

Número de labels:
32


In [23]:
with h5py.File(TRAIN_OUTPUT) as train:

    print(train["images"]["tile_0"].shape)

(640, 640, 3)


In [24]:
with h5py.File(TRAIN_OUTPUT) as train:

    print(train["labels"]["tile_0"][:5])

[[0.         0.90693974 0.12056424 0.05131025 0.05090399]
 [0.         0.7446759  0.3924262  0.05923219 0.06067879]
 [0.         0.7940069  0.44304618 0.05134306 0.05371504]
 [0.         0.8495759  0.02300945 0.04877043 0.04525295]
 [0.         0.7768024  0.50431406 0.07125463 0.07652674]]


## 5. Gerar data.yaml

In [25]:
from pathlib import Path
import yaml

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

data = {
    "path": "/dev/shm/dataset",
    "train": "train/images",
    "val": "val/images",
    "names": {
        0: "tree"
    }
}

with open(DATA_DIR / "data.yaml", "w") as f:
    yaml.dump(
        data,
        f,
        sort_keys=False
    )

print("data.yaml criado com sucesso.")

data.yaml criado com sucesso.
